In [ ]:
import pandas as pd

### 1.Read Raw Excel File

In [ ]:
import os
raw_path = r'recent_sales.csv' if os.path.exists(r'recent_sales.csv') else r'D:\Pankaj\Abu-Dhabi-Pipeline-main\Abu-Dhabi-Pipeline-main\AbuDhabi_RawData.csv'
df = pd.read_csv(raw_path)
print(f'Loaded: {df.shape}')
print(df.head(2))


### 2.Drop Duplicates

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()
df.shape

In [ ]:
df.columns

## 3. Property Type Mapping

In [ ]:
PROPERTY_TYPE_MAP = {
    "apartment": "Flat",
    "duplex": "Flat",
    "residential complex": "Flat",
    "residential complex (staff accommodation)": "Flat",

    "villa": "Villa",
    "townhouse / attached villa": "Villa",
    "palace": "Villa",

    "office": "Office",
    "office complex": "Office",

    "retail": "Shop",
    "mall / market / retail center": "Shop",

    "other commercial plot": "Plot",
    "other industrial plot": "Plot",
    "other mixed-use plot": "Plot",
    "other plot": "Plot",
    "other residential plot": "Plot",
    "other temporary plot": "Plot",
    "plot for clinic / hospital": "Plot",
    "plot for cooling utility": "Plot",
    "plot for farm": "Plot",
    "plot for hotel / resort / serviced apartments": "Plot",
    "plot for mall / market / retail center": "Plot",
    "plot for office complex": "Plot",
    "plot for palace": "Plot",
    "plot for park": "Plot",
    "plot for parking": "Plot",
    "plot for petrol station": "Plot",
    "plot for residential complex": "Plot",
    "plot for residential complex (staff accommodation)": "Plot",
    "plot for school": "Plot",
    "plot for townhouse / attached villa": "Plot",
    "plot for villa": "Plot",

    "clinic / hospital": "Others",
    "cooling utility": "Others",
    "factory": "Others",
    "farm": "Others",
    "hotel / resort / serviced apartments": "Others",
    "mosque": "Others",
    "other": "Others",
    "penthouse": "Others",
    "petrol station": "Others",
    "school": "Others",
    "workshop": "Others"
}

## 4. Project Type Mapping

In [ ]:
PROJECT_TYPE_MAP = {
    "agricultural": "Agricultural",
    "commercial": "Commercial",
    "educational": "Educational",
    "healthcare": "Healthcare",
    "industrial & storage": "Industrial",
    "infrastructural": "Infrastructure",
    "other": "Others",
    "recreational": "Recreational",
    "religious": "Religious",
    "residential": "Residential"
}

## 5. Normalized Unit Configuration Mapping


In [ ]:
NORMALIZED_UNIT_CONFIG_MAP = {
    "1 bed": "1Bhk",
    "2 beds": "2Bhk",
    "3 beds": "3Bhk",
    "4 beds": ">3Bhk",
    "5 beds": ">3Bhk",
    "5+ beds": ">3Bhk",
    "6+ beds": ">3Bhk",
    "anchor store": "Shop",
    "large (500 to 1000 sqm)": "Office",
    "line store": "Shop",
    "maxi (over 1000 sqm)": "Office",
    "medium (50 to 500 sqm)": "Office",
    "small (up to 50 sqm)": "Office",
    "studio": "Flat",
    "unclassified": "Others",

    "1 Bed": "1Bhk",
    "2 Beds": "2Bhk",
    "3 Beds": "3Bhk",
    "4 Beds": ">3Bhk",
    "5 Beds": ">3Bhk",
    "5+ Beds": ">3Bhk",
    "6+ Beds": ">3Bhk",
    "Anchor Store": "Shop",
    "Large (500 To 1000 Sqm)": "Office",
    "Line Store": "Shop",
    "Maxi (Over 1000 Sqm)": "Office",
    "Medium (50 To 500 Sqm)": "Office",
    "Small (Up To 50 Sqm)": "Office",
    "Studio": "Flat",
    "Unclassified": "Others"
}


## 6. Standard Transformation Pipeline


In [ ]:
import pandas as pd
import numpy as np

def transform_abudhabi_transactions(df):

    df = df.copy()

    # =====================================================
    # Date Fields
    # =====================================================
    df["transaction_date"] = pd.to_datetime(
        df["Sale Application Date"],
        format="%Y-%m-%d",
        errors="coerce"
    )

    df["date_of_agreement_execution"] = df["transaction_date"]

    # Extract Year
    df["year"] = df["transaction_date"].dt.year.astype("Int64")

    # Extract Quarter (Q1-2026, Q2-2026, etc.)
    df["quarter"] = (
        "Q"
        + df["transaction_date"].dt.quarter.astype("Int64").astype(str)
        + "-"
        + df["transaction_date"].dt.year.astype("Int64").astype(str)
    )

    # =====================================================
    # Direct Mapping
    # =====================================================
    df["project_name"] = df["Project Name"]

    df["location_name"] = df["District"]

    df["agreement_price"] = pd.to_numeric(
        df["Property Sale Price (AED)"],
        errors="coerce"
    )

    df["net_carpet_area_sq_m"] = pd.to_numeric(
        df["Property Sold Area (SQM)"],
        errors="coerce"
    )

    df["project_stage"] = df["Sale Application Type"]

    df["furnishing_status"] = None

    # =====================================================
    # Property Type
    # =====================================================
    df["property_type_raw"] = (
        df["Property Type"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["property_type"] = (
        df["property_type_raw"]
        .map(PROPERTY_TYPE_MAP)
        .fillna("Others")
    )

    # =====================================================
    # Unit Configuration
    # =====================================================
    df["unit_configuration"] = (
        df["Property Layout"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["property_layout"] = df["Property Layout"]

    # =====================================================
    # Normalized Unit Configuration
    # =====================================================
    df["normalized_unit_configuration"] = (
        df["unit_configuration"]
        .map(NORMALIZED_UNIT_CONFIG_MAP)
        .fillna("Others")
    )

    # =====================================================
    # Project Type
    # =====================================================
    df["project_type"] = (
        df["Asset Class"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(PROJECT_TYPE_MAP)
    )

    # =====================================================
    # Locality Fields
    # =====================================================
    df["sub_locality"] = df["Community"]

    # If Community field exists
    if "Community" in df.columns:
        df["micro_market"] = df["Community"]
    else:
        df["micro_market"] = None

    # =====================================================
    # Property Description
    # =====================================================
    df["property_description"] = (
        df["Property Type"].fillna("").astype(str)
        + " | "
        + df["Asset Class"].fillna("").astype(str)
    )

    # =====================================================
    # Fixed Values
    # =====================================================
    df["transaction_category"] = "Sale"

    df["country_name"] = "UAE"

    df["state_name"] = "Abu Dhabi"

    df["data_source"] = "Adrec"

    df["source_accessibility"] = "Download"

    df["source_accessibility_way"] = "Easy"
    df["city_name"] = "Abu Dhabi"

    # Example:
    # First Sale / Resale
    if "Sale Sequence" in df.columns:
        df["sale_type"] = df["Sale Sequence"]
    else:
        df["sale_type"] = None

    # =====================================================
    # Default DB1 Fields (Not Available)
    # =====================================================
    default_cols = [
        "registered_document_village_name",
        "sub_registrar_office_code",
        "sub_registrar_office_name",
        "document_number",
        "guideline_value",
        "floor_number",
        "unit_number",
        "balcony_sq_m",
        "terrace_sq_m",
        "seller_name",
        "buyer_name",
        "internal_document_number",
        "micr_number",
        "bank_type",
        "party_code",
        "stamp_duty_paid",
        "registration_fee",
        "project_latitude",
        "project_longitude",
        "location_latitude",
        "location_longitude",
        "buyer_pincode",
        "buyer_locality",
        "buyer_district",
        "buyer_state",
        "is_llm_processed",
        "is_manual_processed",
        "tower_name",
        "is_duplicate",
        "pincode",
        "parking_count",
        "facing_direction",
        "view_type",
        "furnishing_status",
        "condition_status",
        "sourcing_cost",
        "sourcing_time",
        "data_type"
    ]

    for col in default_cols:
        df[col] = None

    # =====================================================
    # Convert Dates to DD-MM-YYYY
    # =====================================================
    df["transaction_date"] = (
        df["transaction_date"]
        .dt.strftime("%d-%m-%Y")
    )

    df["date_of_agreement_execution"] = (
        df["date_of_agreement_execution"]
        .dt.strftime("%d-%m-%Y")
    )

    return df

In [ ]:
df_final = transform_abudhabi_transactions(df)

In [ ]:
DB1_COLUMNS = [
    "project_name",
    "location_name",
    "registered_document_village_name",
    "agreement_price",
    "transaction_date",
    "net_carpet_area_sq_m",
    "sub_registrar_office_code",
    "sub_registrar_office_name",
    "document_number",
    "transaction_type",
    "city_name",
    "guideline_value",
    "property_description",
    "floor_number",
    "unit_number",
    "property_type_raw",
    "property_type",
    "year",
    "balcony_sq_m",
    "terrace_sq_m",
    "seller_name",
    "buyer_name",
    "transaction_category",
    "internal_document_number",
    "micr_number",
    "bank_type",
    "party_code",
    "date_of_agreement_execution",
    "stamp_duty_paid",
    "registration_fee",
    "project_latitude",
    "project_longitude",
    "location_latitude",
    "location_longitude",
    "quarter",
    "unit_configuration",
    "property_layout",
    "buyer_pincode",
    "buyer_locality",
    "buyer_district",
    "buyer_state",
    "is_llm_processed",
    "is_manual_processed",
    "tower_name",
    "is_duplicate",
    "sale_type",
    "project_type",
    "country_name",
    "state_name",
    "micro_market",
    "sub_locality",
    "pincode",
    "parking_count",
    "facing_direction",
    "view_type",
    "furnishing_status",
    "condition_status",
    "source_accessibility",
    "source_accessibility_way",
    "sourcing_cost",
    "sourcing_time",
    "data_type",
    "data_source",
    "normalized_unit_configuration",
    "project_stage"
]

In [ ]:
for col in DB1_COLUMNS:
    if col not in df_final.columns:
        df_final[col] = None

In [ ]:
df_db1 = df_final[DB1_COLUMNS]

In [ ]:
import os
cleaned_path = r'recent_sales_cleaned.csv'
# os.makedirs('downloads', exist_ok=True)  # Removed: no local folder creation
df_db1.to_csv(cleaned_path, index=False)
print(f'Cleaned data saved: {df_db1.shape}')


In [ ]:
df_final = pd.read_csv(r'recent_sales_cleaned.csv')


### 7. Location Lat/Long Mapping

In [ ]:
import pandas as pd
import os

raw_path = r'recent_sales_cleaned.csv'
loc_path = r'Abudhabi_location_coordinates.csv'

# 1. Load files
df_raw = pd.read_csv(raw_path)
df_loc = pd.read_csv(loc_path)

# 2. Normalize location strings
df_raw['norm_loc'] = df_raw['location_name'].astype(str).str.strip().str.lower()
df_loc['norm_loc'] = df_loc['location_name'].astype(str).str.strip().str.lower()

# 3. Create location lookup maps
loc_id_map = df_loc.drop_duplicates('norm_loc').set_index('norm_loc')['location_id']
loc_lat_map = df_loc.drop_duplicates('norm_loc').set_index('norm_loc')['location_latitude']
loc_lng_map = df_loc.drop_duplicates('norm_loc').set_index('norm_loc')['location_longitude']

# 4. Map coordinates and location_id
df_raw['location_id'] = df_raw['norm_loc'].map(loc_id_map)
df_raw['location_latitude'] = pd.to_numeric(df_raw['norm_loc'].map(loc_lat_map), errors='coerce')
df_raw['location_longitude'] = pd.to_numeric(df_raw['norm_loc'].map(loc_lng_map), errors='coerce')
df_raw['city_id'] = 1
df_raw['city_name'] = 'Abu Dhabi'

# 5. Fallback for Jabal Hafeet
df_raw.loc[df_raw['norm_loc'] == 'jabal hafeet', ['location_latitude', 'location_longitude']] = [24.0716702, 55.7985914]

df_raw.drop(columns=['norm_loc'], inplace=True)
df_raw.to_csv(raw_path, index=False)
print(f'✅ Location Lat/Long mapped: {df_raw["location_latitude"].notna().sum()}/{len(df_raw)} rows')
print(f'✅ Location IDs mapped: {df_raw["location_id"].notna().sum()}/{len(df_raw)} rows')


### 8. Project Lat/Long Mapping

In [ ]:
# For private projects, fallback to location coordinates
priv_mask = df_raw['norm_proj'] == 'private'
df_raw.loc[priv_mask & df_raw['project_latitude'].isna(), 'project_latitude'] = df_raw.loc[priv_mask, 'location_latitude']
df_raw.loc[priv_mask & df_raw['project_longitude'].isna(), 'project_longitude'] = df_raw.loc[priv_mask, 'location_longitude']

df_raw.drop(columns=['norm_proj', 'norm_loc', 'norm_city', 'key_3attr'], errors='ignore', inplace=True)
df_raw.to_csv(raw_path, index=False)
print(f'🎉 Final Project Lat/Long resolved: {df_raw["project_latitude"].notna().sum()}/{len(df_raw)} rows')


### 9. Assigning Project_id

In [ ]:
import numpy as np
import pandas as pd
import os

raw_path = r'recent_sales_cleaned.csv'
proj_path = r'Abudhabi_project_coordinates.csv'
loc_path = r'Abudhabi_location_coordinates.csv'

df_raw = pd.read_csv(raw_path)
df_proj = pd.read_csv(proj_path)
df_loc = pd.read_csv(loc_path)

df_proj_ref = df_proj.merge(df_loc[['location_id', 'location_name']], on='location_id', how='left')
df_proj_ref['city_name'] = 'Abu Dhabi'

for d in [df_raw, df_proj_ref]:
    d['norm_proj'] = d['project_name'].astype(str).str.strip().str.lower()
    d['norm_loc'] = d['location_name'].astype(str).str.strip().str.lower()
    d['norm_city'] = d['city_name'].astype(str).str.strip().str.lower()

df_raw['match_key'] = df_raw['norm_proj'] + '|' + df_raw['norm_loc'] + '|' + df_raw['norm_city']
df_proj_ref['match_key'] = df_proj_ref['norm_proj'] + '|' + df_proj_ref['norm_loc'] + '|' + df_proj_ref['norm_city']

# 1. Map existing project_id from master table
proj_id_map = df_proj_ref.drop_duplicates('match_key').set_index('match_key')['project_id']
df_raw['project_id'] = df_raw['match_key'].map(proj_id_map)

# 2. Assign sequential IDs starting from 368 for all unlisted projects (including Private)
unlisted_mask = df_raw['project_id'].isna() & (df_raw['norm_proj'] != '') & (df_raw['norm_proj'] != 'nan')
unlisted_projects = df_raw.loc[unlisted_mask, ['project_name', 'match_key']].drop_duplicates()

next_id = int(df_proj['project_id'].max()) + 1
new_id_map = {}
for _, row in unlisted_projects.iterrows():
    new_id_map[row['match_key']] = next_id
    next_id += 1

df_raw['project_id'] = df_raw['project_id'].fillna(df_raw['match_key'].map(new_id_map))

# Reorder columns: place project_id first
cols = ['project_id'] + [c for c in df_raw.columns if c != 'project_id' and not c.startswith('norm_') and c != 'match_key']
df_raw = df_raw[cols]
df_raw.to_csv(raw_path, index=False)

print(df_raw[['project_id', 'project_name', 'location_name']].head(10))
print(f'\n✅ Projects assigned IDs: {df_raw["project_id"].notna().sum()}/{len(df_raw)} rows')
print(f'✅ Private projects assigned IDs: {(df_raw["project_name"].str.lower() == "private").sum()} rows')


### 10. Assign Non Rera Index (internal_index_id) Starting from nr468


In [ ]:
import pandas as pd
import numpy as np
import re
import os

raw_path = r'recent_sales_cleaned.csv'
proj_path = r'Abudhabi_project_coordinates.csv'
loc_path = r'Abudhabi_location_coordinates.csv'

df = pd.read_csv(raw_path)
df_proj = pd.read_csv(proj_path)
df_loc = pd.read_csv(loc_path)

# 1. Enrich master project reference table
df_proj_ref = df_proj.merge(df_loc[['location_id', 'location_name']], on='location_id', how='left')
df_proj_ref['city_name'] = 'Abu Dhabi'

def clean_nr(x):
    if pd.isna(x): return None
    x = str(x).strip()
    x = re.sub(r'__.*$', '', x)
    m = re.match(r'(?i)^nr(\d+)', x)
    return f'nr{m.group(1)}' if m else x

df_proj_ref['norm_proj'] = df_proj_ref['project_name'].astype(str).str.strip().str.lower()
df_proj_ref['norm_loc'] = df_proj_ref['location_name'].astype(str).str.strip().str.lower()
df_proj_ref['norm_city'] = df_proj_ref['city_name'].astype(str).str.strip().str.lower()
df_proj_ref['clean_index'] = df_proj_ref['internal_index_id'].map(clean_nr)

master_index_map = {}
for _, r in df_proj_ref.iterrows():
    k = (r['norm_loc'], r['norm_proj'], r['norm_city'])
    master_index_map[k] = r['clean_index']

# 2. Extract highest NR number in master table
nums = df_proj['internal_index_id'].dropna().str.extract(r'(?i)^nr(\d+)')[0].dropna().astype(int)
max_nr = int(nums.max())
next_num = max_nr + 1  # 468
print(f'Master table highest NR: Nr{max_nr}__abu_dhabi. New projects start at: nr{next_num}')

df['norm_proj'] = df['project_name'].astype(str).str.strip().str.lower()
df['norm_loc'] = df['location_name'].astype(str).str.strip().str.lower()
df['norm_city'] = df.get('city_name', 'Abu Dhabi').astype(str).str.strip().str.lower()
if 'index' not in df.columns:
    df['index'] = None

# 3. Map existing projects from master table
for i in df.index:
    k = (df.at[i, 'norm_loc'], df.at[i, 'norm_proj'], df.at[i, 'norm_city'])
    if k in master_index_map:
        df.at[i, 'index'] = master_index_map[k]

# 4. Assign new sequential NR indexes for newly launched commercial projects (keeping Private blank)
existing_indexes = set(master_index_map.values())
file_mapping = {}
new_nr_count = 0

for i in df.index:
    if pd.isna(df.at[i, 'index']):
        p_norm = df.at[i, 'norm_proj']
        # Keep Private and blank project rows blank
        if p_norm == 'private' or pd.isna(p_norm) or p_norm == '' or p_norm == 'nan':
            continue
        key = (df.at[i, 'norm_loc'], p_norm, df.at[i, 'norm_city'])
        if key in file_mapping:
            df.at[i, 'index'] = file_mapping[key]
        else:
            while f'nr{next_num}' in existing_indexes:
                next_num += 1
            new_idx = f'nr{next_num}'
            df.at[i, 'index'] = new_idx
            file_mapping[key] = new_idx
            existing_indexes.add(new_idx)
            next_num += 1
            new_nr_count += 1

# Explicitly keep Private rows as NaN
df.loc[df['norm_proj'] == 'private', 'index'] = np.nan

df.drop(columns=['norm_proj', 'norm_loc', 'norm_city'], errors='ignore', inplace=True)
df.to_csv(r'recent_sales_cleaned.csv', index=False)
print(f'✅ New NR indexes assigned: {new_nr_count} distinct commercial projects')
print(f'✅ Commercial projects with index: {df["index"].notna().sum()}/{(df["project_name"].str.lower() != "private").sum()} rows')
print(f'✅ Standalone Private rows left as Blank: {(df["project_name"].str.lower() == "private").sum()} rows')


### 11. All Data converted into Tiltle Case

In [ ]:
# Final column order matching standard Abu Dhabi schema
column_order = [
    'internal_index_id', 'project_id', 'project_name', 'location_id', 'location_name',
    'registered_document_village_name', 'agreement_price', 'transaction_date',
    'net_carpet_area_sq_m', 'sub_registrar_office_code', 'sub_registrar_office_name',
    'document_number', 'transaction_type', 'city_id', 'city_name',
    'guideline_value', 'property_description', 'floor_number', 'unit_number',
    'property_type_raw', 'property_type', 'year', 'balcony_sq_m', 'terrace_sq_m',
    'seller_name', 'buyer_name', 'transaction_category', 'internal_document_number',
    'micr_number', 'bank_type', 'party_code', 'date_of_agreement_execution',
    'stamp_duty_paid', 'registration_fee', 'project_latitude', 'project_longitude',
    'location_latitude', 'location_longitude', 'quarter', 'unit_configuration',
    'property_layout', 'buyer_pincode', 'buyer_locality', 'buyer_district',
    'buyer_state', 'is_llm_processed', 'is_manual_processed', 'tower_name',
    'is_duplicate', 'sale_type', 'project_type', 'country_name', 'state_name',
    'micro_market', 'sub_locality', 'pincode', 'parking_count', 'facing_direction',
    'view_type', 'furnishing_status', 'condition_status', 'source_accessibility',
    'source_accessibility_way', 'sourcing_cost', 'sourcing_time', 'data_type',
    'data_source', 'normalized_unit_configuration', 'project_stage'
]
for c in column_order:
    if c not in df.columns:
        df[c] = None
df = df[column_order + [c for c in df.columns if c not in column_order]]

for col in df.select_dtypes(include=['object', 'string']).columns:
    if col not in ['internal_index_id', 'index']:
        df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) and x.lower() not in ['none', 'nan', '<na>'] else x)

df.to_excel(r'recent_sales_processed.xlsx', index=False)
df.to_csv(r'recent_sales_processed.csv', index=False)
print('🎉 Saved Processed CSV and Excel to downloads/ directory!')
